In [1]:
from datasets import load_dataset
from dotenv import load_dotenv, find_dotenv
import pinecone
from pinecone import Pinecone, ServerlessSpec
import os
from sentence_transformers import SentenceTransformer

C:\Users\kubte\anaconda3\envs\vector_db_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
fw = load_dataset("HuggingFaceFW/fineweb", name = "sample-10BT", split = "train", streaming = True)

C:\Users\kubte\anaconda3\envs\vector_db_env\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\kubte\.cache\huggingface\hub\datasets--HuggingFaceFW--fineweb. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [3]:
fw

IterableDataset({
    features: ['text', 'id', 'dump', 'url', 'date', 'file_path', 'language', 'language_score', 'token_count'],
    num_shards: 15
})

In [4]:
fw.features

{'text': Value('string'),
 'id': Value('string'),
 'dump': Value('string'),
 'url': Value('string'),
 'date': Value('string'),
 'file_path': Value('string'),
 'language': Value('string'),
 'language_score': Value('float64'),
 'token_count': Value('int64')}

### ***Embedding*

In [5]:
model = SentenceTransformer("all-MiniLM-L6-v2")

C:\Users\kubte\anaconda3\envs\vector_db_env\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\kubte\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
C:\Users\kubte\anaconda3\envs\vector_db_env\Lib\site-packages\huggingface_hub\file_dow

In [6]:
load_dotenv(find_dotenv(), override = True)

True

In [7]:
pc = Pinecone(
    api_key = os.environ.get("PINECONE_API_KEY")
)

In [8]:
pc.list_indexes()

IndexList([<name='my-index2', ready=True>, <name='my-index', ready=True>])

In [9]:
pc.create_index(
    name = "text",
    dimension = model.get_sentence_embedding_dimension(),
    metric = "cosine",
    spec = ServerlessSpec(
        cloud = "aws",
        region = "us-east-1"
    )
)

C:\Users\kubte\AppData\Local\Temp\ipykernel_19048\2873447656.py:3: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dimension = model.get_sentence_embedding_dimension(),


Name:,text
Status:,Ready
Ready:,Yes
Deployment:,Managed (aws/us-east-1)
Host:,https://text-e9us1v6.svc.aped-4627-b74a.pinecone.io
Deletion Protection:,disabled
Schema fields:,2
Read capacity:,"ReadCapacityOnDemandResponse(status=ReadCapacityStatus(state='Ready', current_shards=None, current_replicas=None, error_message=None))"


In [10]:
index = pc.Index(name = "text")

In [12]:
subset_size = 10000

vectors_to_upsert = []
for i, item in enumerate(fw):
    if i < subset_size:
        break

    text = item['text']
    unique_id = str(item['id'])
    language = item['language']

    embedding = model.encode(text, show_progress_bar = False).tolist()

    metadata = {'language': language}

    vectors_to_upsert.append((unique_id, embedding, metadata))

batch_size = 1000
for i in range(0, len(vectors_to_upsert), batch_size):
    batch = vectors_to_upsert[i:i + batch_size]
    index.upsert(vectors = batch)

print("Subset of data upserted to Pinecone index")

in loop
Subset of data upserted to Pinecone index
